In [ ]:
import sys
import os
import platform
import importlib
import glob

import numpy as np
import pandas as pd
import datetime
import netCDF4

if platform.system() == 'Darwin':  # macOS
    base_FP = '/Users/hyunglokkim/Insync/hkim@geol.sc.edu/Google_Drive'
    cpuserver_data_FP = '/Users/hyunglokkim/cpuserver_data'
elif platform.system() == 'Linux': #workstation
    base_FP = '/home/subin/data'
    cpuserver_data_FP = '/home/subin/cpuserver_data'
else:
    base_FP = '/data'
    cpuserver_data_FP = '/data'
sys.path.append(base_FP + '/python_modules')
print(base_FP)
print(cpuserver_data_FP)

#hydroAI libs
import HydroAI.SMOS_IC as hSMOS_IC
import HydroAI.Grid as hGrid
import HydroAI.Plot as hPlot
import HydroAI.Data as hData
importlib.reload(hSMOS_IC);
importlib.reload(hPlot);
importlib.reload(hData);

# Ignore runtime warnings
import warnings
warnings.filterwarnings("ignore")

# Define your directory where to save nc files
nc_save_dir = cpuserver_data_FP + '/extracted_nc'

## 1. Import and save SMOS_IC SM

In [ ]:
year = 2022
path = 'ASC'
directory = os.path.join(cpuserver_data_FP, 'SMOS_IC', path)
file_list, data_doy = hSMOS_IC.extract_filelist_doy(directory, year) 
longitude, latitude = hGrid.generate_lon_lat_e2grid('25km')

In [ ]:
# Examples (import SMOS-IC data with specific year and overpass time)
SMOS_SM = hSMOS_IC.create_array_from_nc(file_list, data_doy, year, 'Soil_Moisture')
SMOS_RFI = hSMOS_IC.create_array_from_nc(file_list, data_doy, year, 'RMSE')
SMOS_SF = hSMOS_IC.create_array_from_nc(file_list, data_doy, year, 'Scene_Flags')

In [ ]:
# Check the imported data
doy = 109
target = SMOS_SM
target[(target<=0) | (target > 0.6)] = np.nan
hPlot.plot_map(longitude, latitude, target[:,:,doy], np.nanmin(target), np.nanmax(target), plot_title='SMOS SM', label_title='VSM', cmap='jet_r')
target = SMOS_RFI
hPlot.plot_map(longitude, latitude, target[:,:,doy], np.nanmin(target), np.nanmax(target), plot_title='SMOS RFI', label_title='RFI (K)', cmap='jet_r')

In [ ]:
path_list = ['ASC', 'DES']
for year in range(2010, 2024): 
    for path in path_list:
        print(year, path)
        directory = os.path.join(cpuserver_data_FP, 'SMOS_IC', path)
        file_list, data_doy = hSMOS_IC.extract_filelist_doy(directory, year)

        # Import Variables
        SMOS_SM = hSMOS_IC.create_array_from_nc(file_list, data_doy, year, 'Soil_Moisture')
        SMOS_SM[SMOS_SM<=0] = np.nan
        SMOS_RFI = hSMOS_IC.create_array_from_nc(file_list, data_doy, year, 'RMSE')
        SMOS_SF  = hSMOS_IC.create_array_from_nc(file_list, data_doy, year, 'Scene_Flags')

        # Create a NetCDF file
        path_time = 'am' if path == 'ASC' else 'pm'
        file_name = f'SMOS_IC_{year}_{path_time}.nc'
        nc_file_name = os.path.join(nc_save_dir, file_name)

        time_suffix = 'AM' if path == 'ASC' else 'PM'

        print(f"Saving: {nc_file_name}")
        hData.create_netcdf_file(
        nc_file=nc_file_name, 
        latitude=latitude, 
        longitude=longitude,
        **{f'SMOS_SM_{time_suffix}': SMOS_SM, 
           f'SMOS_RFI_{time_suffix}': SMOS_RFI,
           f'SMOS_SF_{time_suffix}': SMOS_SF})

        print(f"NetCDF file created: {nc_file_name}")

## 2. Resample SMOS-IC data (with QC)
Masked data if one of the following conditions was met (Kim et al. (2023)):
1) SM was smaller than 0 cm3cm− 3 higher than 0.60 cm3cm− 3
2) the Tb-RMSE flag values were larger than 5 K.
3) or the science flag was greater than one,

In [ ]:
target_projection = 'Noah_025'
SMOS_lon, SMOS_lat = hGrid.generate_lon_lat_e2grid('25km')
domain_lon, domain_lat = hGrid.generate_lon_lat_eqdgrid(0.25)

domain_lon = domain_lon[:600, :]
domain_lat = domain_lat[:600, :]

path_list = ['am', 'pm']
for year in range(2010, 2024):
    
    for path in path_list:
        time_suffix = 'AM' if path == 'am' else 'PM'
        # resample SMOS into the doamin size
        nc_file  = nc_save_dir+'/SMOS_IC_'+str(year)+'_'+path+'.nc'
        print(nc_file)
        nc_data  = netCDF4.Dataset(nc_file)
        SMOS_SM  = nc_data.variables['SMOS_SM_'+time_suffix][:].data
        
        SMOS_SM_QC = SMOS_SM.copy()
        SMOS_RFI = nc_data.variables['SMOS_RFI_'+time_suffix][:].data
        SMOS_SF  = nc_data.variables['SMOS_SF_'+time_suffix][:].data
        nc_data.close()
        
        # Masking data
        SMOS_SM_QC[(SMOS_SM <= 0) | (SMOS_SM > 0.6)] = np.nan
        SMOS_SM_QC[SMOS_RFI > 5] = np.nan
        SMOS_SM_QC[SMOS_SF  > 1] = np.nan
        
        Resampled_SMOS_SM    = hData.Resampling_forloop(domain_lon, domain_lat, SMOS_lon, SMOS_lat, SMOS_SM)
        Resampled_SMOS_SM_QC = hData.Resampling_forloop(domain_lon, domain_lat, SMOS_lon, SMOS_lat, SMOS_SM_QC)
        Resampled_SMOS_RFI    = hData.Resampling_forloop(domain_lon, domain_lat, SMOS_lon, SMOS_lat, SMOS_RFI)
        Resampled_SMOS_SF = hData.Resampling_forloop(domain_lon, domain_lat, SMOS_lon, SMOS_lat, SMOS_SF)
        
        file_name = 'SMOS_IC_'+str(year)+'_'+path+'_R_'+target_projection+'.nc'
        nc_file_name = nc_save_dir+'/'+file_name

        data_vars = {}
        data_vars['Resampled_SMOS_SM'] = Resampled_SMOS_SM
        data_vars['Resampled_SMOS_SM_QC'] = Resampled_SMOS_SM_QC
        data_vars['Resampled_SMOS_RFI'] = Resampled_SMOS_RFI
        data_vars['Resampled_SMOS_SF'] = Resampled_SMOS_SF

        print(f"Saving: {nc_file}")
        hData.create_netcdf_file(
        nc_file    = nc_file_name,
        latitude   = domain_lat,
        longitude  = domain_lon,
        **data_vars)

        print(f"NetCDF file created: {nc_file}")

In [ ]:
doy = 150
target = SMOS_SM[:,:, doy]
hPlot.plot_map(SMOS_lon, SMOS_lat, target, np.nanmin(target), np.nanmax(target), plot_title='SMOS SM', label_title='-', cmap='jet_r')
target = SMOS_SM_QC[:,:,doy]
hPlot.plot_map(SMOS_lon, SMOS_lat, target, np.nanmin(target), np.nanmax(target), plot_title='SMOS SM QC', label_title='-', cmap='jet_r')
target = Resampled_SMOS_SM_QC[:,:,doy]
hPlot.plot_map(domain_lon, domain_lat, target, np.nanmin(target), np.nanmax(target), plot_title='Resampled SMOS SM QC', label_title='-', cmap='jet_r')